# DATASCI 151 – Final Project
## Formula One Racing Analysis (1950–Present)

**Group Members:**
- Emily Huisman: 2669072
- Erin Thompson: **PUT YOUR STUDENT ID**
- Tess Gilmartin: **PUT YOUR STUDENT ID**

<font color='red'> A lot of the text i wrote were gonna need to rewrite just wrote it quickly so yall knew what i did</font>

## Introduction

<font color='red'> TODO: A markdown cell with 1-2 paragraphs that summarize the main goals of the
project. Imagine you are giving a report to someone who is not your professor.
The first paragraph should briefly describe what the topic is, what data analysis
question you’re interested in, and why it is relevant. The introduction should end
with a high-level summary of the results and the coming structure of the project.
Try to make the text self-contained, intended for someone who isn’t familiar with
Formula 1/FIFA/NFL/U.S. politics/another topic or the dataset being used.
</font>



## Research Questions

1. **Grid vs. Outcome:** Does starting grid position predict race finishing position? Has this relationship changed across different decades in F1?
2. **Nationality Analysis:** Which nationalities have produced the most successful drivers? How has this shifted over time?
3. **Driver Age & Performance:** How does a driver's points per race change as they age? Is there a peak age in F1?

---
## Section 1: Data Description

### 1.1 Libraries & Data Import

We will use the following five CSV files from the Formula One dataset to answer our questions. 
 - `results.csv`: one row per driver per race, recording grid position, finishing position, and points earned
 - `races.csv`: one row per race with the year, round number, and circuit
 - `drivers.csv`: one row per driver with name, nationality, and date of birth
 - `constructors.csv`: one row per constructor (team) with name and nationality
 - `status.csv`: is a lookup table decoding finish status codes (e.g., "Finished", "Engine", "Accident").

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

: 

In [9]:
# You guys will need to change "data_path" to make sure it works for you
# I just created a folder on my desktop with this noteboook and the 5 csv files
data_path = '/final_project/1-Formula_One'  

results_df      = pd.read_csv(data_path + 'results.csv')
races_df        = pd.read_csv(data_path + 'races.csv')
drivers_df      = pd.read_csv(data_path + 'drivers.csv')
constructors_df = pd.read_csv(data_path + 'constructors.csv')
status_df       = pd.read_csv(data_path + 'status.csv')

# Looking at the shape of each data set
print('results:', results_df.shape)
print('races:', races_df.shape)
print('drivers:', drivers_df.shape)
print('constructors:', constructors_df.shape)
print('status:', status_df.shape)

: 

### 1.2 Merging
We're going to merge all five tables into one dataframe using the result table as it contains a key that matches to each other table that we can merge on. Using left joing to ensure we keep all race information. <font color='red'> FIX WORDING/MAKE BETTER </font>

In [30]:
# Merging results and races
df = results_df.merge(races_df[['raceId', 'year', 'round', 'name', 'circuitId']],on='raceId', how='left')

# Merging that with drivers
df = df.merge(drivers_df[['driverId', 'forename', 'surname', 'nationality', 'dob']], on='driverId', how='left')

# Merging constructors
# rename before merge to avoid error with existing race name column 
df = df.merge(constructors_df[['constructorId', 'name']].rename(columns={'name': 'constructor_name'}), on='constructorId', how='left')
# Step 4: + status
df = df.merge(status_df, on='statusId', how='left')

print('Merged dataframe shape:', df.shape)
df.head(10)

Merged dataframe shape: (25660, 28)


,resultId,raceId,driverId,constructorId,number,grid,position,positionText,positionOrder,points,...,year,round,name,circuitId,forename,surname,nationality,dob,constructor_name,status
0,1,18,1,1,22,1,1,1,1,10.0,...,2008,1,Australian Grand Prix,1,Lewis,Hamilton,British,1985-01-07,McLaren,Finished
1,2,18,2,2,3,5,2,2,2,8.0,...,2008,1,Australian Grand Prix,1,Nick,Heidfeld,German,1977-05-10,BMW Sauber,Finished
2,3,18,3,3,7,7,3,3,3,6.0,...,2008,1,Australian Grand Prix,1,Nico,Rosberg,German,1985-06-27,Williams,Finished
3,4,18,4,4,5,11,4,4,4,5.0,...,2008,1,Australian Grand Prix,1,Fernando,Alonso,Spanish,1981-07-29,Renault,Finished
4,5,18,5,1,23,3,5,5,5,4.0,...,2008,1,Australian Grand Prix,1,Heikki,Kovalainen,Finnish,1981-10-19,McLaren,Finished
5,6,18,6,3,8,13,6,6,6,3.0,...,2008,1,Australian Grand Prix,1,Kazuki,Nakajima,Japanese,1985-01-11,Williams,+1 Lap
6,7,18,7,5,14,17,7,7,7,2.0,...,2008,1,Australian Grand Prix,1,Sébastien,Bourdais,French,1979-02-28,Toro Rosso,Engine
7,8,18,8,6,1,15,8,8,8,1.0,...,2008,1,Australian Grand Prix,1,Kimi,Räikkönen,Finnish,1979-10-17,Ferrari,Engine
8,9,18,9,2,4,2,\N,R,9,0.0,...,2008,1,Australian Grand Prix,1,Robert,Kubica,Polish,1984-12-07,BMW Sauber,Collision
9,10,18,10,7,12,18,\N,R,10,0.0,...,2008,1,Australian Grand Prix,1,Timo,Glock,German,1982-03-18,Toyota,Accident


### 1.3 Data Cleaning

In [37]:
# Need to replace '\N' for missing values with NaN so that pandas knows its missing data
df.replace('\\N', np.nan, inplace=True)

# figure out object types
#print(df.dtypes)

# convert necessary columns to numeric
for column in ['grid', 'position', 'points', 'laps', 'milliseconds']: 
    df[column] = pd.to_numeric(df[column])

# parse date of birth so we can use it later to calculate age
df['dob'] = pd.to_datetime(df['dob'])

# create a column with full name
df['driver_name'] = df['forename'] + ' ' + df['surname']

# create a column thats true if driver finished
df['finished'] = df['status'] == 'Finished'

# create decade column
df['decade'] = pd.cut(df['year'], 
                      bins = [1949, 1959, 1969, 1979, 1989, 1999, 2009, 2019, 2029],
                      labels = ['1950s','1960s','1970s','1980s','1990s','2000s','2010s','2020s'])

### 1.4 Descriptive Statistics

<!-- TODO: Write a short paragraph describing the main columns you'll use.
     Mention: grid, position, points, nationality, year, dob.
     Reference the table output below. -->

In [38]:
descriptive_cols = ['grid', 'position', 'points', 'year']
df[descriptive_cols].describe().round(2)

,grid,position,points,year
count,25660.00,14833.00,25660.00,25660.00
mean,11.19,7.93,1.85,1990.03
std,7.25,4.80,4.13,19.23
min,0.00,1.00,0.00,1950.00
25%,5.00,4.00,0.00,1976.00
50%,11.00,7.00,0.00,1990.00
75%,17.00,11.00,2.00,2007.00
max,34.00,33.00,50.00,2022.00


In [39]:
# Races per decade
df.groupby('decade')['raceId'].nunique()

/var/folders/0c/d28_phtd6fl6133s0dd8ct840000gn/T/ipykernel_75490/1553049338.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby('decade')['raceId'].nunique()


decade
1950s     84
1960s    100
1970s    144
1980s    156
1990s    162
2000s    174
2010s    198
2020s     52
Name: raceId, dtype: int64

---
## Section 2: Results

### Question 1 — Does Grid Position Predict Race Outcome, and Has This Changed Over Time?

<!-- TODO: Write 1-2 sentences here introducing what you're doing in this section -->

In [1]:
# Filter to only rows where both grid and finishing position are available
# Overall correlation between grid and finishing position
# Scatter plot grid position vs. finishing position
# correlation by decade and plot (line plot)
# 



<!-- TODO: Write 2-3 sentences interpreting your Q1 findings.
     What is the overall correlation? Did grid position become more or less predictive over time?
     Any interesting decades (e.g., turbo era 1980s, hybrid era 2010s)? -->

---
### Question 2 — Which Nationalities Produce the Most Successful Drivers, and How Has This Shifted Over Time?

<!-- TODO: Write 1-2 sentences introducing this section -->

In [ ]:
# Total wins by nationality (all time) ---
# Bar chart — top 10 nationalities by total race wins
# Wins by nationality over time (decade breakdown) - plot as line chart
# could also do points per race by nationallity so like avg opints per entry by nationality


<!-- TODO: Write 2-3 sentences interpreting Q2 findings.
     Which nationalities dominate overall? Any interesting trends over decades?
     E.g., British dominance early, Brazilian surge in the 80s-90s, German dominance 2000s? -->

---
### Question 3 — How Does Driver Performance Change With Age?

<!-- TODO: Write 1-2 sentences introducing this section -->

In [2]:
# Compute driver age at time of each race - use race date - write a function that computes age in years use dob and race date
# avg points per race by age
# plot avg points vs age
# 

<!-- TODO: Write 2-3 sentences interpreting Q3 findings.
     What is the peak age? How sharp is the drop-off after peak?
     Any surprising findings (very young or old high-performers)? -->

---
## Section 3: Discussion

